# 1장 — 큰 그림: 안개 낀 산과 여행자 🏔️

웹 챕터의 지형을 코드로 다시 만들어 **출발점이 운명을 가르는 것**을 재현하고,
**안개의 이유**(조합 폭발)를 숫자로 확인합니다.

**이 노트북의 목표**
1. 2D toy 손실 지형에서 경사하강: 출발점 3곳의 서로 다른 운명(전역·국소·평원)
2. 등고선 지도(신의 시점) 위에 세 여행의 자취 그리기
3. 지형 전체를 그리는 비용이 왜 불가능한 숫자인지 계산

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch1')

## 1. 지형 준비: 웹과 같은 toy 손실 지형

전역 골짜기(깊음, (7,4) 근처) + 국소최소(얕음, (2.5,7) 근처) + 완만한 그릇.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def L(x, y):
    """손실 지형 — 가중치 2개(x, y)의 함수. 실제 모델은 이 함수가 수십억 차원."""
    r1 = (x - 7)**2 + (y - 4)**2
    r2 = (x - 2.5)**2 + (y - 7)**2
    return 3 - 2.5*np.exp(-r1/3) - 1.2*np.exp(-r2/2) + 0.02*((x - 5)**2 + (y - 5)**2)

def grad(x, y):
    """발밑 경사(그래디언트) — 2부 8장의 편미분을 모은 것. 여행자가 아는 전부."""
    r1 = (x - 7)**2 + (y - 4)**2
    r2 = (x - 2.5)**2 + (y - 7)**2
    e1, e2 = np.exp(-r1/3), np.exp(-r2/2)
    gx = (5/3)*(x - 7)*e1 + 1.2*(x - 2.5)*e2 + 0.04*(x - 5)
    gy = (5/3)*(y - 4)*e1 + 1.2*(y - 7)*e2 + 0.04*(y - 5)
    return gx, gy

def hike(x, y, lr=0.8, steps=200):
    """경사하강 하산 — 발밑 경사만 보고 걷는다 (난수 없음, 완전 결정적)."""
    path = [(x, y)]
    for _ in range(steps):
        gx, gy = grad(x, y)
        x, y = x - lr*gx, y - lr*gy
        path.append((x, y))
    return np.array(path)

print('지형 준비 완료. 전역최소 고도:', f'{L(6.95, 4.02):.3f}', '/ 국소최소 고도:', f'{L(2.58, 6.93):.3f}')

## 2. 실험 ①: 출발점 셋, 운명 셋

In [ ]:
starts = {'🏜️ 북동쪽 평원': (10.3, 8.8), '⛰️ 서쪽 능선': (1.5, 8.2), '🌄 남쪽 기슭': (6.2, 1.5)}
paths = {name: hike(*p) for name, p in starts.items()}

# 신의 시점 — 등고선 지도 위에 세 자취
gx_, gy_ = np.meshgrid(np.linspace(0, 11, 120), np.linspace(0, 10, 110))
plt.figure(figsize=(7, 5))
plt.contourf(gx_, gy_, L(gx_, gy_), levels=25, cmap='viridis_r')
plt.colorbar(label='손실 (고도)')
for (name, path), c in zip(paths.items(), ['tab:orange', 'tab:red', 'white']):
    plt.plot(path[:, 0], path[:, 1], '.-', ms=3, lw=1, color=c, label=name)
plt.scatter([6.95], [4.02], marker='*', s=180, c='gold', edgecolors='k', label='전역최소 🚩', zorder=5)
plt.legend(loc='lower left'); plt.title('안개 낀 산: 신의 시점과 세 여행의 자취')
plt.tight_layout(); plt.show()

for name, path in paths.items():
    x15, y15 = path[15]
    xf, yf = path[-1]
    print(f'{name}: 15걸음 고도 {L(x15, y15):.2f} → 200걸음 고도 {L(xf, yf):.3f}  도착 ({xf:.2f},{yf:.2f})')

plateau, ridge, south = (paths[k] for k in starts)
assert L(*south[15]) < 0.7, '남쪽 기슭: 몇 걸음 만에 전역 골짜기(0.60 근처) 도착'
assert abs(L(*ridge[-1]) - 2.0) < 0.1, '서쪽 능선: 국소최소(고도 2.0)에 정착 — 더 깊은 곳을 모른 채'
assert L(*plateau[15]) > 3.0, '북동쪽 평원: 15걸음에도 거의 못 내려옴 (발밑이 평평해 기어감)'
assert L(*plateau[-1]) < 0.7, '…하지만 200걸음이면 평원 출발도 결국 전역최소 도착'

# 국소최소의 잔인함 — 그 지점의 발밑 경사는 정말 0이다
gm = np.hypot(*grad(*ridge[-1]))
assert gm < 1e-3, '국소최소의 발밑 경사 ≈ 0 — 여행자에겐 전역최소와 똑같이 느껴진다'
print(f'\n✔ 확인: 같은 지형·같은 다이얼, 출발점만 달라도 운명이 갈린다 (국소최소 |∇L| = {gm:.5f})')

## 3. 실험 ② · 안개의 이유: 조합 폭발

지형 전체를 그리려면 가중치마다 10칸 격자만 훑어도 **10^가중치 수**의 손실 계산이 필요합니다.

In [ ]:
import time

# W = 2 (오늘의 toy): 실제로 다 계산해 본다 — 즉시 끝
t0 = time.perf_counter()
gridN = sum(1 for a in np.linspace(0, 11, 10) for b in np.linspace(0, 10, 10) if L(a, b) >= 0)
dt = time.perf_counter() - t0
print(f'W=2 : 10^2 = {gridN}번 계산, {dt*1000:.1f}ms (그래서 신의 시점을 보여드릴 수 있었다)')

# W가 커지면? — 자릿수만 계산해 본다
for W, note in [(10, '겨우 10개'), (10**9, '실제 LLM급')]:
    digits = W  # 10^W 의 자릿수-1 = W
    print(f'W={W:<12,}: 10^{W:,} 번, 1 뒤에 0이 {digits:,}개 ({note})')
print('참고: 우주의 원자 수 ≈ 10^80. W=81 짜리 미니 모델의 지도조차 우주가 못 담는다')

assert gridN == 100, '10×10 격자 = 100번'
print('\n✔ 확인: 안개는 게으름이 아니라 수학. 그래서 발밑 기울기(역전파 1번)만 보며 걷는다')

## 4. 관찰 정리 + 🤖 AI 연결

- **손실 지형** = 가중치 조합을 위치로, 손실을 고도로 놓은 것. 훈련은 이 지형의 하산.
- 여행자는 **안개** 속: 지형 전체는 조합 폭발(10^W)로 영원히 못 보고,
  발밑 경사(그래디언트: 2부 8장 편미분의 모음, 역전파 한 번)만 안다.
- 지물들: **전역/국소최소**(여행자는 현장에서 구분 불가, |∇L| = 0 은 똑같다),
  **평원**(걸음 = 보폭 × 기울기라 기울기 ≈ 0 이면 제자리걸음), 가파른 사면(폭발 위험).
- 출발점(가중치 초기화)도 운명을 가른다. 그리고 이 모든 조건에서 "잘 걷는 기술"이 8부의 다이얼들.

**다음 챕터**: 2장(보폭의 과학, 첫 핵심). 워밍업의 그 경계(1.00)가 왜 하필 1인지,
보폭과 지형의 **가파름(곡률)** 사이의 정확한 관계를 계산으로 밝혀냅니다.